# NumPy 数组与数值计算

第五讲实验：怎样保持样本配对不变。

先阅读 lab.md 的运行与预期结果说明。所有房屋数据都是合成教学数据，单位为平方米、房间数和万元。只需 CPU 与 NumPy，不需网络或 API。先逐格练习，最终重启内核并按顺序运行全部。不要把保留的旧输出当作你本次执行的证据。

## 1 记录环境

导入 NumPy 并记录版本。版本信息便于解释行为与计时差异，不代表不同版本必然失败。

In [1]:
import sys
import numpy as np
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

Python: 3.12.14
NumPy: 2.3.5


## 2 输入的数值 单位 与形状

每个位置代表同一套房。先预测 area.shape，再运行。

In [2]:
area = np.array([50., 60., 80., 90.], dtype=np.float64)
target = np.array([110., 130., 170., 190.], dtype=np.float64)
print("shape, ndim, size, dtype:", area.shape, area.ndim, area.size, area.dtype)
assert area.shape == target.shape == (4,)
print("列表重复:", [50, 60] * 2)
print("数组逐元素乘:", np.array([50, 60]) * 2)

shape, ndim, size, dtype: (4,) 1 4 float64
列表重复: [50, 60, 50, 60]
数组逐元素乘: [100 120]


## 3 先写明循环参考

模型 C 为 1.5 × 面积 + 40。手算预测应为 115、130、160、175 万元。

In [3]:
loop_prediction = []
for value in area:
    loop_prediction.append(1.5 * float(value) + 40.0)
loop_prediction = np.array(loop_prediction)
print("循环预测:", loop_prediction)

循环预测: [115. 130. 160. 175.]


## 4 逐元素计算 与逐项核验

先核对形状，再比较数值。把中间误差保留下来，避免只看最后的均值。

In [4]:
array_prediction = 1.5 * area + 40.0
assert array_prediction.shape == loop_prediction.shape == (4,)
np.testing.assert_allclose(array_prediction, loop_prediction, rtol=0, atol=1e-12)
signed_error = array_prediction - target
absolute_error = np.abs(signed_error)
print("数组预测:", array_prediction)
print("带符号误差:", signed_error)
print("绝对误差:", absolute_error)
print("MAE / 万元:", absolute_error.mean())
assert absolute_error.mean() == 7.5

数组预测: [115. 130. 160. 175.]
带符号误差: [  5.   0. -10. -15.]
绝对误差: [ 5.  0. 10. 15.]
MAE / 万元: 7.5


## 5 重现既定规则比较

先前课程已经选定 B；这里复核训练和已披露的测试结果，不根据测试标签更改规则。

In [5]:
rules = {"A": (2., 0.), "B": (2., 10.), "C": (1.5, 40.)}
scores = {}
for name, (weight, bias) in rules.items():
    predictions = weight * area + bias
    scores[name] = float(np.abs(predictions - target).mean())
print("训练 MAE / 万元:", scores)
assert scores == {"A": 10., "B": 0., "C": 7.5}
test_area = np.array([65., 85.])
test_target = np.array([144., 174.])
test_mae = np.abs(2. * test_area + 10. - test_target).mean()
baseline_mae = np.abs(target.mean() - test_target).mean()
print("既定测试 B 与基线 MAE:", test_mae, baseline_mae)
assert test_mae == 5. and baseline_mae == 15.

训练 MAE / 万元: {'A': 10.0, 'B': 0.0, 'C': 7.5}
既定测试 B 与基线 MAE: 5.0 15.0


## 6 三个样本 两个特征

X 的列依次为面积和房间数。整数索引与范围切片可能保留不同轴。

In [6]:
X = np.array([[50., 1.], [60., 2.], [80., 1.]])
for name, values in [("X", X), ("X[0]", X[0]), ("X[:1]", X[:1]),
                     ("X[:,0]", X[:,0]), ("X[:,:1]", X[:,:1])]:
    print(name, "shape=", values.shape, "values=", values.tolist())
assert X.shape == (3, 2)
assert X[:1].shape == (1, 2)
assert X[:,:1].shape == (3, 1)

X shape= (3, 2) values= [[50.0, 1.0], [60.0, 2.0], [80.0, 1.0]]
X[0] shape= (2,) values= [50.0, 1.0]
X[:1] shape= (1, 2) values= [[50.0, 1.0]]
X[:,0] shape= (3,) values= [50.0, 60.0, 80.0]
X[:,:1] shape= (3, 1) values= [[50.0], [60.0], [80.0]]


## 7 归约的是哪条轴

先使用无单位数字看清方向。axis=0 合并样本轴；axis=1 合并特征轴。

In [7]:
Z = np.array([[1., 2.], [3., 4.], [5., 6.]])
print("sum axis=0:", Z.sum(axis=0), Z.sum(axis=0).shape)
print("sum axis=1:", Z.sum(axis=1), Z.sum(axis=1).shape)
print("全部均值:", Z.mean(), Z.mean().shape)
print("keepdims axis=0:", Z.mean(axis=0, keepdims=True).shape)
print("keepdims axis=1:", Z.mean(axis=1, keepdims=True).shape)
np.testing.assert_array_equal(Z.sum(axis=0), [9, 12])
np.testing.assert_array_equal(Z.sum(axis=1), [3, 7, 11])

sum axis=0: [ 9. 12.] (2,)
sum axis=1: [ 3.  7. 11.] (3,)
全部均值: 3.5 ()
keepdims axis=0: (1, 2)
keepdims axis=1: (3, 1)


## 8 广播每个特征的系数

面积系数为 2 万元每平方米，房间系数为 5 万元每间，公共偏置 10 万元。系数是指定的教学规则，没有拟合过程。

In [8]:
weights = np.array([2., 5.])
contribution = X * weights
weighted_prediction = contribution.sum(axis=1) + 10.
print("贡献 / 万元:\n", contribution)
print("预测 / 万元:", weighted_prediction)
assert contribution.shape == (3, 2)
assert weighted_prediction.shape == (3,)
np.testing.assert_allclose(weighted_prediction, [115., 140., 175.], rtol=0, atol=1e-12)

贡献 / 万元:
 [[100.   5.]
 [120.  10.]
 [160.   5.]]
预测 / 万元: [115. 140. 175.]


## 9 双层循环是另一条核验路线

不要只依赖两个相似的数组表达式互相证明。循环与手算给出独立的对应关系。

In [9]:
weighted_reference = []
for row in X:
    total = 10.0
    for value, weight in zip(row, weights):
        total += float(value) * float(weight)
    weighted_reference.append(total)
weighted_reference = np.array(weighted_reference)
assert weighted_reference.shape == weighted_prediction.shape
np.testing.assert_allclose(weighted_prediction, weighted_reference, rtol=0, atol=1e-12)
print("循环与数组逐项一致:", weighted_reference)

循环与数组逐项一致: [115. 140. 175.]


## 10 故意制造广播陷阱

下面两个数组装着相同的三个数，正确配对误差为零。猜测列状预测减一维标签会产生什么形状。

In [10]:
p = np.array([115., 140., 175.])
y = p.copy()
wrong_difference = p[:, None] - y
print("正确 MAE:", np.abs(p-y).mean())
print("错误形状:", wrong_difference.shape)
print("错误差值:\n", wrong_difference)
print("错误 MAE:", np.abs(wrong_difference).mean())
assert wrong_difference.shape == (3, 3)
assert np.isclose(np.abs(wrong_difference).mean(), 80/3, rtol=0, atol=1e-12)

正确 MAE: 0.0
错误形状: (3, 3)
错误差值:
 [[  0. -25. -60.]
 [ 25.   0. -35.]
 [ 60.  35.   0.]]
错误 MAE: 26.666666666666668


## 11 先检查接口 再求平均

完整评分函数在 experiment.py 中。导入不会自动运行整个实验。这里的异常是预期测试，必须被明确捕获；如果没有拒绝，测试反而失败。

In [11]:
from experiment import mae_checked, weighted_predict, require_value_error
print("正常评分:", mae_checked(p, y))
try:
    mae_checked(p[:, None], y)
except ValueError as error:
    print("正确拒绝列状预测:", error)
else:
    raise AssertionError("Expected a shape error")
require_value_error(lambda: mae_checked([1., 2.], [1.]))
require_value_error(lambda: mae_checked([], []))
print("长度不符和空输入也已被拒绝")

正常评分: 0.0
正确拒绝列状预测: prediction and target must both have shape (n,)
长度不符和空输入也已被拒绝


## 12 非方阵 单样本 单特征

批处理接口应在一个样本时仍保留二维输入和一维输出。调用前后比较原数据，确认函数没有修改输入。

In [12]:
before = X.copy()
normal = weighted_predict(X, weights, 10.)
one = weighted_predict(X[:1], weights, 10.)
one_feature = weighted_predict(X[:,:1], [2.], 10.)
assert normal.shape == (3,)
assert one.shape == (1,) and one[0] == 115.
np.testing.assert_array_equal(one_feature, [110., 130., 170.])
np.testing.assert_array_equal(X, before)
print("非方阵:", normal, "单样本:", one, "单特征:", one_feature)

非方阵: [115. 140. 175.] 单样本: [115.] 单特征: [110. 130. 170.]


## 13 应该拒绝的输入

识别错误与算出正确值同样属于函数行为。每个测试都对应一条具体契约。

In [13]:
bad_cases = [
    ("丢掉样本轴", lambda: weighted_predict(X[0], weights, 10.)),
    ("系数长度错误", lambda: weighted_predict(X, [2.], 10.)),
    ("偏置不是标量", lambda: weighted_predict(X, weights, [10.])),
    ("没有样本", lambda: weighted_predict(np.empty((0, 2)), weights, 10.)),
    ("包含 nan", lambda: weighted_predict([[np.nan, 1.]], weights, 10.)),
    ("数字写成字符串", lambda: weighted_predict([["50", "1"]], weights, 10.)),
]
for name, operation in bad_cases:
    require_value_error(operation)
    print("已拒绝:", name)

已拒绝: 丢掉样本轴
已拒绝: 系数长度错误
已拒绝: 偏置不是标量
已拒绝: 没有样本
已拒绝: 包含 nan
已拒绝: 数字写成字符串


## 14 视图与副本

修改前先预测 a、view、copy、selected 各自会怎样改变。不要在主实验 X 上直接做这一步。

In [14]:
a = np.array([50., 60., 80., 90.])
view = a[1:3]
copy = a[1:3].copy()
selected = a[[1, 2]]
print("是否共享:", np.shares_memory(a, view), np.shares_memory(a, copy), np.shares_memory(a, selected))
view[0] = 600.
print("原数组:", a, "视图:", view, "副本:", copy, "高级索引读取:", selected)
assert a[1] == 600. and copy[0] == selected[0] == 60.

是否共享: True False False
原数组: [ 50. 600.  80.  90.] 视图: [600.  80.] 副本: [60. 80.] 高级索引读取: [60. 80.]


## 15 类型范围与赋值

这些小实验故意触发数值风险；不要用于金额计算。先转换再算，与先算错再转换，是两回事。

In [15]:
narrow = np.array([120], dtype=np.int8)
wrapped = narrow + np.array([10], dtype=np.int8)
safe = narrow.astype(np.int64) + 10
integer_values = np.array([1, 2], dtype=np.int64)
integer_values[0] = 1.9
print("int8 溢出:", wrapped, "先转 int64:", safe)
print("整数数组保存 1.9 后:", integer_values)
assert int(wrapped[0]) == -126 and int(safe[0]) == 130
assert integer_values[0] == 1

int8 溢出: [-126] 先转 int64: [130]
整数数组保存 1.9 后: [1 2]


## 16 累加精度与信息损失

第一组差异发生在累加时；第二组差异在数值进入低精度数组时已经发生。

In [16]:
v = np.array([100000000., 1., -100000000.], dtype=np.float32)
print("默认 float32 累加:", v.sum())
print("指定 float64 累加:", v.sum(dtype=np.float64))
already_rounded = np.array([100000001.], dtype=np.float32).astype(np.float64)
print("先低精度保存再转换:", already_rounded)
assert float(v.sum()) == 0.
assert float(v.sum(dtype=np.float64)) == 1.
assert already_rounded[0] == 100000000.
print("精确相等:", (0.1 + 0.2) == 0.3)
np.testing.assert_allclose([0.1 + 0.2], [0.3], rtol=0, atol=1e-12)
print("明确容差的比较通过")

默认 float32 累加: 0.0
指定 float64 累加: 1.0
先低精度保存再转换: [1.e+08]
精确相等: False
明确容差的比较通过


## 17 用规模估算内存 不创建巨型数组

只算元素数与字节数。意外交叉差的存储随样本数平方增长。

In [17]:
for n in [1000, 10000, 100000]:
    paired_bytes = 8 * n
    crossed_bytes = 8 * n * n
    print("n=", n, "配对字节=", paired_bytes, "错误交叉字节=", crossed_bytes)

n= 1000 配对字节= 8000 错误交叉字节= 8000000
n= 10000 配对字节= 80000 错误交叉字节= 800000000
n= 100000 配对字节= 800000 错误交叉字节= 80000000000


## 18 先正确 再计时

输入准备不计入时间；两边均执行预测与 MAE。每种规模预热后测量五组，报告每次调用时间中位数。数值因机器与负载而变，不是固定答案。

In [18]:
from experiment import timing_experiment
measurements = timing_experiment()
for record in measurements:
    print("n={n:6d}  循环={loop_ms_per_call:.6f} ms  数组={array_ms_per_call:.6f} ms".format(**record))

n=     4  循环=0.000707 ms  数组=0.004401 ms
n=   100  循环=0.008837 ms  数组=0.004638 ms
n= 10000  循环=0.747179 ms  数组=0.017295 ms
n=100000  循环=8.421297 ms  数组=0.621078 ms


## 19 完整数值与边界检查

这里调用同目录脚本的检查函数，不包含重复计时。它覆盖已在前面理解的数值、形状、异常与不修改输入行为。

In [19]:
from experiment import run_checks
report = run_checks(include_timing=False)
assert report["status"] == "passed"
assert len(report["checks"]) == 10
print("状态:", report["status"], "检查组数:", len(report["checks"]))
for check in report["checks"]:
    print(check["name"], check["passed"])
print("关键结果:", report["results"])

状态: passed 检查组数: 10
house_predictions_loop_array_and_mae True
fixed_test_b_5_and_training_mean_baseline_15 True
non_square_3_by_2_prediction_and_no_mutation True
single_sample_retains_batch_axis True
single_feature_retains_feature_axis True
axis_reductions_and_keepdims True
broadcast_bug_reproduced_and_contract_rejects_it True
invalid_dimensions_empty_nonfinite_strings_booleans_rejected True
slice_view_explicit_copy_and_advanced_index_copy True
dtype_overflow_truncation_accumulation_and_tolerance True
关键结果: {'training_mae_wan': {'A': 10.0, 'B': 0.0, 'C': 7.5}, 'test_b_mae_wan': 5.0, 'test_baseline_mae_wan': 15.0, 'weighted_prediction_wan': [115.0, 140.0, 175.0], 'broadcast_wrong_mae_wan': 26.666666666666668, 'sum_float32': 0.0, 'sum_float64_accumulator': 1.0}


## 20 写下自己的解释

在新文字格完成以下句子：

1. (n,1) 与 (n,) 相减会产生……，因为……。
2. 我的非方阵与单样本测试分别帮助发现……。
3. 本次计时包含……，排除了……，所以结论只适用于……。

然后重启内核并运行全部，确认没有未处理异常。练习题与详解见 lecture.md 和 answers.md。